### Understanding TFT data and working with .json files for analysis


### Placement & Performance
- What is my average placement?
- What is my top 4 rate? (placement ≤ 4)
- What is my first place rate?
- How has my average placement trended over time — am I improving?
- What is my worst loss streak / best win streak?
### Game Length & Level
- What is my most common ending level?
- What is the average game length in minutes?
- Do games where I hit level 9 lead to better placements?
- Is there a correlation between level and placement?
### Units & Comps
- What are my most played units?
- Which units give me the best average placement?
- Which units do I play when I finish 1st vs 8th?
- What is my most played 3-star unit?
### Traits & Synergies
- What are my most activated traits?
- Which traits correlate with top 4 finishes?
- What is my most played comp/archetype?
### Augments
- Which augments do I pick most often?
- Which augments give me the best average placement?
- Which augments do I consistently underperform with?
### Patterns & Habits
- What time of day do I play most?
- Do I perform better in longer or shorter sessions?
- How many games do I play per day on average?

In [2]:
import json
import pandas as pd

with open("data/matches.json") as f:
    matches = json.load(f)

print(f"Loaded {len(matches)} matches")

Loaded 50 matches


### Extract stats into dataframe

In [3]:
def extract_my_stats(matches):
    rows = []
    for match in matches:
        puuid = match["_my_puuid"]
        info = match["info"]
        me = next((p for p in info["participants"] if p["puuid"] == puuid), None)
        if me is None:
            continue
        rows.append({
            "match_id":  match["metadata"]["match_id"],
            "date":      pd.to_datetime(info["game_datetime"], unit="ms"),
            "placement": me["placement"],
            "level":     me["level"],
            "augments":  [a.replace("TFT_Augment_", "") for a in me.get("augments", [])],
            "traits":    me.get("traits", []),
            "units":     me.get("units", []),
        })
    return pd.DataFrame(rows)

df = extract_my_stats(matches)
df.head()

,match_id,date,placement,level,augments,traits,units
0,NA1_5562880650,2026-05-18 04:53:11.410,4,9,[],"[{'name': 'TFT17_APTrait', 'num_units': 4, 'st...","[{'character_id': 'TFT17_Aatrox', 'itemNames':..."
1,NA1_5560691122,2026-05-15 05:37:09.012,6,9,[],"[{'name': 'TFT17_AnimaSquad', 'num_units': 1, ...","[{'character_id': 'TFT17_Nasus', 'itemNames': ..."
2,NA1_5559979852,2026-05-14 05:08:10.272,4,9,[],"[{'name': 'TFT17_APTrait', 'num_units': 4, 'st...","[{'character_id': 'TFT17_Aatrox', 'itemNames':..."
3,NA1_5559669327,2026-05-13 23:18:42.862,5,9,[],"[{'name': 'TFT17_AnimaSquad', 'num_units': 1, ...","[{'character_id': 'TFT17_IvernMinion', 'itemNa..."
4,NA1_5559167035,2026-05-13 03:01:23.733,5,8,[],"[{'name': 'TFT17_ADMIN', 'num_units': 2, 'styl...","[{'character_id': 'TFT17_Teemo', 'itemNames': ..."


In [4]:
df_csv = df.copy()
df_csv["augments"] = df_csv["augments"].apply(json.dumps)
df_csv["traits"] = df_csv["traits"].apply(json.dumps)
df_csv["units"] = df_csv["units"].apply(json.dumps)

df_csv.to_csv("data/tft_games.csv", index=False)
print(f"Saved {len(df_csv)} rows to data/tft_games.csv")

Saved 50 rows to data/tft_games.csv
